In [ ]:
!pip install -q comet_ml

### Inicialización y Configuración de Comet ML

In [ ]:
import comet_ml
import os
import numpy as np
from google.colab import userdata

# Configuración de credenciales de Comet
# Opcion 1: Setear LOCALMENTE
# os.environ["COMET_API_KEY"] = "TU_API_KEY_AQUI"

# Opcion 2: Setear en Colab Secrets
# os.environ["COMET_API_KEY"] = userdata.get('COMET_API_KEY')

## Función Genérica para Inicializar Experimentos
Definimos una función robusta y reutilizable que automatiza la creación del experimento, la asignación de un nombre estructurado, el registro de parámetros clave solicitados en la consigna y la asignación de tags para filtrado rápido en la interfaz web de Comet.

In [ ]:
def init_comet_experiment(config, run_version=1):
    """
    Inicializa un experimento en Comet ML con una nomenclatura estructurada
    y registra de forma transparente la configuración del modelo y del dataset.
    """

    WORKSPACE = "kaggle-taa-freesound-audio-tagging"
    PROJECT_NAME = "trained_cnns"  # Un unico proyecto tendremos

    # Generar un nombre de experimento descriptivo y paramétrico
    # Formato: [Tipo_De_Modelo]_[Arquitectura]_[Dataset]_v[Versión]
    if config["fine_tuning"]:
        prefix = "FT"
    elif config["transfer_learning"]:
        prefix = "TL"
    else:
        prefix = "Baseline"

    experiment_name = f"{prefix}_{config['architecture_name']}_{config['train_dataset_type']}_v{run_version}"

    print(f"==> Inicializando Experimento Comet ML: '{experiment_name}'")

    # Crear el objeto de experimento
    experiment = comet_ml.Experiment(
        api_key=os.environ.get("COMET_API_KEY"),
        project_name=PROJECT_NAME,
        workspace=WORKSPACE,
        auto_histogram_weight_logging=True,
        auto_histogram_gradient_logging=True,
        auto_histogram_activation_logging=True
    )

    # Asignar el nombre en el dashboard
    experiment.set_name(experiment_name)

    # Registrar todos los parámetros de configuración (Hyperparameters)
    experiment.log_parameters(config)
    experiment.log_parameter("run_version", run_version)

    # Asignar los tags estructurales según la estrategia de MLOps definida
    tags = [prefix.lower(), config["architecture_name"], config["train_dataset_type"]]
    if config.get("data_augmentation", False):
        tags.append("data-augmentation")
    if config.get("is_final_candidate", False):
        tags.append("final-candidate")

    experiment.add_tags(tags)

    return experiment

## Ejemplos de Configuración segun Casos de la Consigna
A continuación, se configuran diccionarios genéricos que representan las diferentes variantes requeridas por la letra del proyecto. Al cambiar el diccionario enviado a la función, Comet creará un registro completamente estructurado.

In [ ]:
# --- CASO A: Red CNN Custom desde cero (Baseline) entrenada solo con datos Curados ---
config_baseline = {
    "architecture_name": "custom_cnn_4layers",
    "transfer_learning": False,
    "fine_tuning": False,
    "train_dataset_type": "curated_only",  # Especifica con qué dataset fue entrenado
    "audio_duration_seconds": 5.0,        # Decisión basada en el EDA
    "spectrogram_type": "mel_spectrogram",
    "data_augmentation": False,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "optimizer": "Adam",
    "is_final_candidate": False
}

# --- CASO B: Transfer Learning con MobileNetV2 usando pesos congelados e incluyendo Data Augmentation ---
config_transfer_learning = {
    "architecture_name": "mobilenetv2",
    "transfer_learning": True,
    "fine_tuning": False,
    "train_dataset_type": "curated_only",
    "audio_duration_seconds": 7.0,
    "spectrogram_type": "mel_spectrogram",
    "data_augmentation": True,           # Variante con aumento de datos
    "batch_size": 64,
    "learning_rate": 1e-4,
    "optimizer": "Adam",
    "is_final_candidate": False
}

# --- CASO C: Propuesta avanzada utilizando datos Noisy como Pre-entrenamiento (Warm-Start) y Fine-Tuning final en Curated ---
config_fine_tuning_noisy = {
    "architecture_name": "efficientnetb0",
    "transfer_learning": True,
    "fine_tuning": True,
    "train_dataset_type": "noisy_pretrain_curated_finetune", # Estrategia para datos ruidosos
    "audio_duration_seconds": 5.0,
    "spectrogram_type": "mel_spectrogram",
    "data_augmentation": True,
    "batch_size": 32,
    "learning_rate": 1e-5,                # LR baja típica de fine-tuning
    "optimizer": "Adam",
    "is_final_candidate": True            # Marcado como fuerte candidato al informe
}

## Simulación de un Ciclo de Entrenamiento y Logging de Métricas
Para probar la infraestructura de Comet ML sin necesidad de descargar el dataset completo de audios, ejecutamos una simulación. Aquí verás cómo registrar la métrica oficial obligatoria: **`lwlrap`**.

In [ ]:
# Simulacion de metricas por clase y matriz de confusion
def evaluate_model_on_validation_set():
    # En un escenario real, esto se calcula con el modelo entrenado sobre el dataset de validación.
    # Aquí devolvemos datos falsos para la simulación.
    clases_simuladas = [f"Clase_{i}" for i in range(80)]
    map_por_clase = {f"mAP_{clase}": np.random.uniform(0.1, 0.9) for clase in clases_simuladas}

    # Simulamos matriz de confusión (reducida a 5 clases para no saturar la salida)
    clases_reducidas = ["Bark", "Meow", "Siren", "Engine", "Wind"]
    y_true = np.random.choice(clases_reducidas, size=100)
    y_pred = np.random.choice(clases_reducidas, size=100)

    return map_por_clase, y_true, y_pred, clases_reducidas

# Seleccionamos uno de los esquemas de configuración creados anteriormente
current_config = config_fine_tuning_noisy
VERSION_EXPERIMENTO = 1

try:
    experiment = init_comet_experiment(current_config, run_version=VERSION_EXPERIMENTO)

    print("\nSimulando épocas de entrenamiento...")
    epochs = 10

    for epoch in range(1, epochs + 1):
        # Aquí iría tu ciclo real con model.fit() o entrenamiento custom
        # Supongamos que calculamos el lwlrap y la pérdida en validación:
        simulated_train_loss = 2.5 / (epoch * 0.8 + 1)
        simulated_val_loss = 2.6 / (epoch * 0.75 + 1) + np.random.normal(0, 0.05)

        # La métrica clave del proyecto: lwlrap (debe tender a 1.0)
        simulated_train_lwlrap = 0.2 + (0.65 * (epoch / epochs))
        simulated_val_lwlrap = 0.18 + (0.60 * (epoch / epochs)) + np.random.normal(0, 0.02)

        # Registro de métricas por época (Aca reemplazar por la historia de la cnn en si)
        experiment.log_metric("loss", simulated_train_loss, step=epoch)
        experiment.log_metric("val_loss", simulated_val_loss, step=epoch)
        experiment.log_metric("lwlrap", simulated_train_lwlrap, step=epoch)
        experiment.log_metric("val_lwlrap", simulated_val_lwlrap, step=epoch)

        print(f"Época {epoch:02d}/{epochs:02d} -> loss: {simulated_train_loss:.4f} | val_lwlrap: {simulated_val_lwlrap:.4f}")

    # --- Al final del entrenamiento: Evaluación detallada ---
    print("\nGenerando métricas de diagnóstico detalladas...")
    map_scores, y_true_val, y_pred_val, labels = evaluate_model_on_validation_set()

    # 1. Registrar métricas por clase (ej: mAP)
    experiment.log_metrics(map_scores)

    # 2. Registrar la Matriz de Confusión
    experiment.log_confusion_matrix(
        y_true=y_true_val,
        y_predicted=y_pred_val,
        labels=labels
    )

    # Al terminar el entrenamiento, finalizamos el experimento explícitamente
    experiment.end()
    print("\n==> ¡Experimento completado y subido con éxito a Comet ML! Checkea el link generado arriba.")

except Exception as e:
    print(f"\n[Aviso] No se pudo conectar a Comet ML automáticamente (¿Falta la API Key?). Error: {e}")
    print("Cuando corras esto en tu cuenta de Colab con tu API Key válida, se registrará de forma perfecta.")